# Adversarial validation: how different are train and test?

`04_missingness_diagnostic.ipynb` found that train and test missingness rates differ by
up to 3.38 percentage points across all twelve columns. This sizes that.

The method: throw away the real target, label every train row 0 and every test row 1,
and train a classifier to tell them apart under the same 5-fold CV. The AUC of that
classifier is a direct measure of how distinguishable the two sets are.

- **AUC near 0.5** means the two sets are interchangeable. The shift is cosmetic, the
  CV scheme needs no adjustment, and fold averaging remains the whole explanation for
  the CV/LB gap.
- **AUC materially above 0.5** means a model can tell where a row came from. The
  features driving that are worth knowing, and training rows that look like test rows
  are worth more than rows that do not.

**This writes no ledger row.** It trains a model, but not a model of the competition
target, so it has no CV score that belongs in a table of competition results.

## What counts as material

With 987,671 rows the standard error on this AUC is tiny, so statistical significance
is not the interesting bar and will be cleared by trivial differences. The practical
bar used here:

- below 0.52: ignore it
- 0.52 to 0.60: real but mild, worth knowing which features carry it
- above 0.60: strong enough that fold weighting or feature removal deserves testing

In [1]:
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
N_SPLITS = 5
TARGET = "addicted_label"
ID = "id"
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]


def locate():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "raw" / "train.csv").exists():
            return base, base / "data" / "raw"
    kag = Path("/kaggle/input/playground-series-s6e8")
    if (kag / "train.csv").exists():
        return Path("/kaggle/working"), kag
    raise FileNotFoundError("could not find train.csv")


REPO, RAW = locate()
train = pd.read_csv(RAW / "train.csv")
test = pd.read_csv(RAW / "test.csv")
FEATURES = [c for c in train.columns if c not in (ID, TARGET)]
print(f"train {len(train):,}  test {len(test):,}  features {len(FEATURES)}")

train 691,369  test 296,302  features 12


## Build the adversarial set

`id` is excluded, and this is not optional here. Train ids run 0 to 691368 and test ids
run 691369 to 987670, so `id` separates the two sets perfectly. Leaving it in would
produce an AUC of 1.0 and measure nothing except that the organizers numbered the rows
in order.

In [2]:
X = pd.concat([train[FEATURES], test[FEATURES]], ignore_index=True)
is_test = np.concatenate([np.zeros(len(train), dtype=int), np.ones(len(test), dtype=int)])

for c in CAT_COLS:
    X[c] = pd.Categorical(X[c])

assert ID not in X.columns, "id separates train from test perfectly and must be excluded"
assert TARGET not in X.columns, "the real target must not leak into this"
assert len(X) == len(train) + len(test)
print(f"{len(X):,} rows, {is_test.mean():.3f} of them test")

987,671 rows, 0.300 of them test


In [3]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof = np.zeros(len(X), dtype=float)
fold_scores = []
importances = np.zeros(len(FEATURES))
t0 = time.time()

for f, (tr, va) in enumerate(skf.split(X, is_test)):
    model = lgb.LGBMClassifier(n_estimators=300, random_state=SEED, verbose=-1)
    model.fit(X.iloc[tr], is_test[tr])
    p = model.predict_proba(X.iloc[va])[:, 1]
    oof[va] = p
    fold_scores.append(roc_auc_score(is_test[va], p))
    importances += model.feature_importances_ / N_SPLITS
    print(f"  fold {f}: auc={fold_scores[-1]:.6f}")

adv_auc = float(np.mean(fold_scores))
adv_std = float(np.std(fold_scores))
print(f"\nadversarial AUC: {adv_auc:.6f} +/- {adv_std:.6f}   ({time.time() - t0:.0f}s)")

  fold 0: auc=0.562160


  fold 1: auc=0.562886


  fold 2: auc=0.562437


  fold 3: auc=0.564383


  fold 4: auc=0.562472

adversarial AUC: 0.562868 +/- 0.000792   (88s)


## Verdict

In [4]:
print(f"adversarial AUC = {adv_auc:.6f}\n")
if adv_auc < 0.52:
    print("VERDICT: train and test are effectively interchangeable.")
    print("The missingness differences are real but too small for a model to exploit.")
    print("No change to the CV scheme. Fold averaging remains the explanation for the")
    print("CV/LB gap, and this line of investigation is closed.")
elif adv_auc < 0.60:
    print("VERDICT: mild but real shift. A model can partially tell the sets apart.")
    print("Worth knowing which features carry it. Fold weighting is probably not worth")
    print("the complexity at this level, but the CV/LB gap now has two explanations")
    print("rather than one and neither should be stated as settled.")
else:
    print("VERDICT: strong shift. Test rows are materially unlike train rows.")
    print("Adversarial weighting or dropping the offending features deserves a real")
    print("experiment, and the CV scheme itself may be mis-specified.")

adversarial AUC = 0.562868

VERDICT: mild but real shift. A model can partially tell the sets apart.
Worth knowing which features carry it. Fold weighting is probably not worth
the complexity at this level, but the CV/LB gap now has two explanations
rather than one and neither should be stated as settled.


## Which features carry the difference

Importance here means "useful for telling train from test", not "useful for predicting
addiction". A feature at the top of this list is one whose distribution moved.

In [5]:
imp = (pd.DataFrame({"feature": FEATURES, "importance": importances})
       .sort_values("importance", ascending=False))
imp["share"] = 100 * imp["importance"] / imp["importance"].sum()
print(f"{'feature':<26}{'importance':>12}{'share%':>9}")
print("-" * 47)
for r in imp.itertuples():
    print(f"{r.feature:<26}{r.importance:>12.0f}{r.share:>9.1f}")

feature                     importance   share%
-----------------------------------------------
sleep_hours                       1043     11.6
work_study_hours                  1006     11.2
gaming_hours                       995     11.1
notifications_per_day              979     10.9
social_media_hours                 970     10.8
daily_screen_time_hours            951     10.6
weekend_screen_time                918     10.2
app_opens_per_day                  916     10.2
age                                590      6.6
stress_level                       251      2.8
academic_work_impact               204      2.3
gender                             177      2.0


## How test-like is the training set?

If the shift is real, the adversarial probability on a training row says how much that
row resembles the test set. Saving it costs nothing and is what any later fold-weighting
experiment would need as an input.

In [6]:
train_adv = oof[:len(train)]
print(f"adversarial probability on training rows:")
for q in [0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99]:
    print(f"  p{int(q * 100):>2}: {np.quantile(train_adv, q):.4f}")
print(f"\nmean {train_adv.mean():.4f}, and a row is 'test-like' above roughly "
      f"{is_test.mean():.4f}")

OOF_DIR = REPO / "artifacts" / "oof"
OOF_DIR.mkdir(parents=True, exist_ok=True)
np.save(OOF_DIR / "adversarial_train_prob.npy", train_adv)
print(f"\nsaved to {OOF_DIR / 'adversarial_train_prob.npy'}")
print("No ledger row written: this models the train/test split, not the competition target.")

adversarial probability on training rows:
  p 1: 0.1759
  p10: 0.2299
  p25: 0.2680
  p50: 0.2946
  p75: 0.3210
  p90: 0.3640
  p99: 0.4431

mean 0.2962, and a row is 'test-like' above roughly 0.3000

saved to E:\Claude\kaggle\comps\smartphone-addiction\artifacts\oof\adversarial_train_prob.npy
No ledger row written: this models the train/test split, not the competition target.
